# Clase 224 — Métricas de fairness: DP, equalized odds, calibration

Dataset sintético binario con atributo protegido `A∈{0,1}` y **base rates diferentes** (necesario para activar el teorema de imposibilidad). Requiere: `pip install numpy pandas scikit-learn matplotlib`. `fairlearn` opcional — implementamos todo a mano.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 10_000

# Atributo protegido (50/50)
A = rng.integers(0, 2, n)

# Features con distribución que depende de A (proxy realista)
x1 = rng.normal(loc=0.5 * A, scale=1.0, size=n)
x2 = rng.normal(loc=-0.3 * A, scale=1.0, size=n)
x3 = rng.normal(loc=0.0, scale=1.0, size=n)

# Base rates DIFERENTES → activa impossibility
logits = 1.2 * x1 - 0.8 * x2 + 0.5 * x3 + np.where(A == 0, 0.4, -0.4)
p = 1 / (1 + np.exp(-logits))
y = (rng.random(n) < p).astype(int)

X = np.column_stack([x1, x2, x3, A])
print(f'n={n} | base rate A=0: {y[A==0].mean():.3f} | base rate A=1: {y[A==1].mean():.3f}')

## 1. Baseline: LogisticRegression

In [ ]:
X_tr, X_te, y_tr, y_te, A_tr, A_te = train_test_split(
    X, y, A, test_size=0.3, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
s = clf.predict_proba(X_te)[:, 1]
yhat = (s >= 0.5).astype(int)

print(f'Accuracy global: {accuracy_score(y_te, yhat):.4f}')
print(f'AUC:             {roc_auc_score(y_te, s):.4f}')
print(f'Acc A=0: {accuracy_score(y_te[A_te==0], yhat[A_te==0]):.4f}')
print(f'Acc A=1: {accuracy_score(y_te[A_te==1], yhat[A_te==1]):.4f}')

## 2. Demographic parity gap

`DP_gap = |P(Ŷ=1|A=0) − P(Ŷ=1|A=1)|`

In [ ]:
def selection_rate(yhat, A, a):
    return yhat[A == a].mean()

sr0 = selection_rate(yhat, A_te, 0)
sr1 = selection_rate(yhat, A_te, 1)
dp_gap = abs(sr0 - sr1)
ratio = min(sr0, sr1) / max(sr0, sr1)

print(f'selection_rate(A=0) = {sr0:.4f}')
print(f'selection_rate(A=1) = {sr1:.4f}')
print(f'DP_gap              = {dp_gap:.4f}')
print(f'80% rule ratio      = {ratio:.4f}  (regla EEOC: >= 0.80)')

## 3. Equal opportunity y equalized odds (Hardt-Price-Srebro 2016)

TPR = P(Ŷ=1 | Y=1, A=a) — `equal opportunity` exige TPR igual.  
FPR = P(Ŷ=1 | Y=0, A=a) — `equalized odds` exige TPR **y** FPR iguales.

In [ ]:
def tpr_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return tpr, fpr

tpr0, fpr0 = tpr_fpr(y_te[A_te == 0], yhat[A_te == 0])
tpr1, fpr1 = tpr_fpr(y_te[A_te == 1], yhat[A_te == 1])

eo_opp_gap = abs(tpr0 - tpr1)
eo_odds_gap = max(abs(tpr0 - tpr1), abs(fpr0 - fpr1))

print(f'TPR A=0 = {tpr0:.4f} | TPR A=1 = {tpr1:.4f}  -> equal_opportunity_gap = {eo_opp_gap:.4f}')
print(f'FPR A=0 = {fpr0:.4f} | FPR A=1 = {fpr1:.4f}')
print(f'equalized_odds_gap = max(|TPR_diff|, |FPR_diff|) = {eo_odds_gap:.4f}')

## 4. Calibration por grupo (Chouldechova 2017)

Para cada bin de score, P(Y=1 | Ŝ in bin) debe coincidir entre grupos.

In [ ]:
def calibration_curve_grouped(y_true, scores, A, a, n_bins=10):
    mask = A == a
    s, y = scores[mask], y_true[mask]
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(s, bins[1:-1])
    mean_score, mean_y = [], []
    for b in range(n_bins):
        m = idx == b
        if m.sum() > 0:
            mean_score.append(s[m].mean())
            mean_y.append(y[m].mean())
    return np.array(mean_score), np.array(mean_y)

ms0, my0 = calibration_curve_grouped(y_te, s, A_te, 0)
ms1, my1 = calibration_curve_grouped(y_te, s, A_te, 1)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect calibration')
ax.plot(ms0, my0, 'o-', label='A=0', color='steelblue')
ax.plot(ms1, my1, 's-', label='A=1', color='darkorange')
ax.set_xlabel('mean predicted score'); ax.set_ylabel('mean y_true')
ax.set_title('Reliability curves por grupo (baseline)')
ax.legend(); ax.grid(alpha=0.3); plt.show()

k = min(len(my0), len(my1))
cal_gap_baseline = np.abs(my0[:k] - my1[:k]).max()
print(f'calibration_gap baseline = {cal_gap_baseline:.4f}  (más bajo = más calibrado entre grupos)')

## 5. Demostración numérica del teorema de imposibilidad

Buscamos `t_0` y `t_1` tales que el modelo cumpla **DP exacta**. Recalculamos predictive parity (PPV) — debe **empeorar**.

In [ ]:
def apply_thresholds(s, A, t0, t1):
    return np.where(A == 0, (s >= t0).astype(int), (s >= t1).astype(int))

best = None
for t0 in np.linspace(0.1, 0.9, 81):
    target_sr = (s[A_te == 0] >= t0).mean()
    t1 = np.quantile(s[A_te == 1], 1 - target_sr)
    yhat_dp = apply_thresholds(s, A_te, t0, t1)
    acc = accuracy_score(y_te, yhat_dp)
    if best is None or acc > best['acc']:
        best = {'t0': t0, 't1': t1, 'acc': acc, 'yhat': yhat_dp}

print(f"DP-mitigated: t0={best['t0']:.3f}, t1={best['t1']:.3f}, acc={best['acc']:.4f}")
sr0_dp = selection_rate(best['yhat'], A_te, 0)
sr1_dp = selection_rate(best['yhat'], A_te, 1)
print(f'DP_gap post-fix = {abs(sr0_dp - sr1_dp):.4f}  (≈0 por construcción)')

def ppv(y_true, y_pred, A, a):
    m = (A == a) & (y_pred == 1)
    return y_true[m].mean() if m.sum() else float('nan')

ppv0_base = ppv(y_te, yhat, A_te, 0); ppv1_base = ppv(y_te, yhat, A_te, 1)
ppv0_dp = ppv(y_te, best['yhat'], A_te, 0); ppv1_dp = ppv(y_te, best['yhat'], A_te, 1)

print('\nPPV (predictive parity proxy) — debe ser igual entre grupos si hay calibración:')
print(f'  baseline:     PPV A=0={ppv0_base:.4f}, PPV A=1={ppv1_base:.4f}, gap={abs(ppv0_base-ppv1_base):.4f}')
print(f'  DP-mitigated: PPV A=0={ppv0_dp:.4f}, PPV A=1={ppv1_dp:.4f}, gap={abs(ppv0_dp-ppv1_dp):.4f}')
print('\n-> Forzar DP rompe predictive parity. Teorema KMR/Chouldechova 2017 en acción.')

## 6. Post-processing Hardt 2016: thresholds que minimizan equalized odds gap

In [ ]:
best_eo = None
for t0 in np.linspace(0.1, 0.9, 41):
    for t1 in np.linspace(0.1, 0.9, 41):
        yhat_eo = apply_thresholds(s, A_te, t0, t1)
        tpr0_, fpr0_ = tpr_fpr(y_te[A_te == 0], yhat_eo[A_te == 0])
        tpr1_, fpr1_ = tpr_fpr(y_te[A_te == 1], yhat_eo[A_te == 1])
        gap = max(abs(tpr0_ - tpr1_), abs(fpr0_ - fpr1_))
        acc = accuracy_score(y_te, yhat_eo)
        if best_eo is None or (gap < best_eo['gap'] - 1e-4) or (abs(gap - best_eo['gap']) < 1e-4 and acc > best_eo['acc']):
            best_eo = {'t0': t0, 't1': t1, 'gap': gap, 'acc': acc, 'yhat': yhat_eo}

print(f"EO-mitigated: t0={best_eo['t0']:.3f}, t1={best_eo['t1']:.3f}, EO_gap={best_eo['gap']:.4f}, acc={best_eo['acc']:.4f}")

## 7. Tabla comparativa

In [ ]:
def evaluate(yhat_, s_, y_te, A_te, name):
    sr0 = selection_rate(yhat_, A_te, 0); sr1 = selection_rate(yhat_, A_te, 1)
    tpr0, fpr0 = tpr_fpr(y_te[A_te == 0], yhat_[A_te == 0])
    tpr1, fpr1 = tpr_fpr(y_te[A_te == 1], yhat_[A_te == 1])
    return {
        'model': name,
        'accuracy': accuracy_score(y_te, yhat_),
        'AUC': roc_auc_score(y_te, s_),
        'DP_gap': abs(sr0 - sr1),
        'EO_opp_gap': abs(tpr0 - tpr1),
        'EO_odds_gap': max(abs(tpr0 - tpr1), abs(fpr0 - fpr1)),
        'PPV_gap': abs(ppv(y_te, yhat_, A_te, 0) - ppv(y_te, yhat_, A_te, 1)),
    }

tabla = pd.DataFrame([
    evaluate(yhat, s, y_te, A_te, 'baseline (t=0.5)'),
    evaluate(best['yhat'], s, y_te, A_te, 'DP-mitigated'),
    evaluate(best_eo['yhat'], s, y_te, A_te, 'EO-mitigated'),
])
print(tabla.round(4).to_string(index=False))

## Ejercicio guiado

1. Reemplazá el dataset sintético por **Adult Census** (UCI). Atributo protegido: `sex`. Target: `income > 50K`.
2. Implementá las 4 métricas a mano sobre el dataset real. ¿Cumple regla del 80%?
3. Repetí la mitigación EO con grid search. ¿Cuánto cae la accuracy?
4. Comparar tu implementación contra `fairlearn.metrics.MetricFrame` y `fairlearn.postprocessing.ThresholdOptimizer`.
5. Bonus: probá COMPAS (ProPublica) con `race`. Reproducí el debate ProPublica vs Northpointe — ¿calibración o equalized odds?

## Conclusiones

- **Demographic parity** ignora el ground truth; útil cuando las base rates son comparables.
- **Equal opportunity / equalized odds** (Hardt 2016) condicionan en Y — más defendibles en crédito, contratación, salud.
- **Calibration / predictive parity** (Chouldechova 2017) asegura que el score signifique lo mismo en cada grupo.
- **Teorema de imposibilidad** (KMR / Chouldechova 2017): si las base rates difieren, no se pueden tener las tres a la vez — hay que **elegir y documentar**.
- Post-processing con thresholds por grupo es la mitigación más simple y la que mejor revela el trade-off accuracy/fairness.